## ReplicaSet — keeping N pods alive

The **ReplicaSet** is the simplest workload controller. Its whole job: keep exactly `N` pods matching a label selector alive.

```yaml
apiVersion: apps/v1
kind: ReplicaSet
metadata: { name: web }
spec:
  replicas: 3
  selector:
    matchLabels: { app: web }
  template:
    metadata:
      labels: { app: web }      # MUST match the selector
    spec:
      containers:
        - name: app
          image: nginx:1.27-alpine
```

Three fields in `spec`: **`replicas`** (target count), **`selector`** (the label query for "my pods"), **`template`** (a full pod spec). The controller loops: count pods matching the selector → too few, create from `template` → too many, delete the excess → forever.

Two things to lock in:

- **Selector and template labels must agree.** If `selector` says `app: web` but the template labels say `app: api`, the controller creates pods it can't see, then loops forever making more. Kubernetes rejects this at admission — but it's a real footgun in custom controllers.
- **Owned pods carry `ownerReferences`** back to the ReplicaSet (and names like `web-x2k5p`). That's how deletes cascade: delete the parent, the pods go too.

**Self-healing:** delete a pod a ReplicaSet owns and a replacement appears within seconds — same template, new name. That's the reconcile loop you can watch live. On our map this is the **ReplicaSet** chip in Workload kinds — the count-keeper. But you rarely write one directly, for the reason the next section explains.